# Lesson 02 - Истраживање Microsoft Agent Framework-а

The **Microsoft Agent Framework (MAF)** је унификован оквир за изградњу AI агената. Пружа чисту, композициону архитектуру са четири основна градивна блока:

- **Client** – повезује се са крајњом тачком AI модела и управља комуникацијом
- **Agent** – омотава клијента са упутствима и дефиницијама алата
- **Tools** – проширују могућности агента прилагођеним функцијама које модел може да позове
- **Session** – одржава историју разговора за интеракције у више корака

У овој лекцији ћемо изградити **агента за резервацију путовања** који проверава доступност одредишта користећи ове концепте.


## Подешавање


In [ ]:
# Install the Microsoft Agent Framework package
! pip install agent-framework azure-ai-projects -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## Разумевање архитектуре агента

Microsoft Agent Framework прати слојевиту архитектуру:

```
Client  →  Agent  →  Tools
                  →  Session
```

1. **Клијент** – `AzureAIProjectAgentProvider` се повезује на Azure OpenAI распоред. Обрађује аутентификацију, форматирање захтева и парсирање одговора.
2. **Агент** – Креиран из клијента преко `provider.create_agent()`, агент комбинује приступ моделу са упутствима (системски подсетник) и алатима.
3. **Алатке** – Python функције украшене са `@tool` које агент може позвати да би извршио радње или преузео податке.
4. **Сесија** – Објекат `AgentSession` (креиран преко `agent.create_session()`) који чува историју разговора, омогућавајући дијалог са више корака где агент памти претходни контекст.

Хајде да изградимо сваки слој корак по корак.


In [ ]:
# Create the client – this is the connection to the AI model
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Додавање алата помоћу @tool декоратора

Алатке омогућавају агентима да предузму радње изван генерисања текста. `@tool` декоратор претвара обичну Python функцију у нешто што агент може да позове.

Кључне тачке:
- Користите `Annotated[type, "description"]` да модел разуме сваки параметар.
- Docstring постаје опис алатке који модел види.
- `approval_mode="never_require"` значи да алатка ради аутоматски без корисничког потврђивања.


In [ ]:
@tool(approval_mode="never_require")
def check_destination_availability(
    destination: Annotated[str, "The destination to check availability for"]
) -> str:
    """Check if a vacation destination is currently available for booking."""
    available = {
        "Barcelona": True,
        "Tokyo": True,
        "Cape Town": False,
        "Vancouver": True,
        "Dubai": False,
    }
    is_available = available.get(destination, False)
    return f"{destination} is {'available' if is_available else 'not available'} for booking."

## Креирање агента са алатима

Сада спајамо клијента, упутства и алате у агента. `instructions` делују као системски подсетник — они дефинишу личност и понашање агента.


In [ ]:
agent = await provider.create_agent(
    name="TravelAvailabilityAgent",
    instructions=(
        "You are a travel booking agent. Help users check destination availability "
        "and make recommendations. Always check availability before recommending a destination."
    ),
    tools=[check_destination_availability],
)

## Вишеокружни разговори са сесијама

`AgentSession` (креиран преко `agent.create_session()`) прати све поруке у разговору. Прослеђивањем исте сесије сваком позиву `agent.run()`, агент има приступ целокупној историји разговора и може да се позове на претходне поруке.

Прослеђујемо `tools=[check_destination_availability]` тако да агент може да позове наш проверивач доступности током сваког корака.


In [ ]:
session = agent.create_session()

# Turn 1: Ask about available destinations
response = await agent.run(
    "Which destinations do you have available?",
    session=session,
)
print(f"Agent: {response}")

In [ ]:
# Turn 2: Follow-up question — the agent remembers the conversation
response = await agent.run(
    "I'd like to go somewhere warm. What's available?",
    session=session,
)
print(f"Agent: {response}")

## Сажетак

У овој лекцији сте истражили четири стуба Microsoft Agent Framework-a:

| Концепт | Оно што сте научили |
|---------|---------------------|
| **Клијент** | `AzureAIProjectAgentProvider` се повезује на Azure OpenAI уз аутентификацију базирану на акредитивима |
| **Агент** | `provider.create_agent()` комбинује везу са моделом са упутствима и именом |
| **Алатке** | `@tool` декоратор омогућава позивање Python функција од стране агента |
| **Сесија** | `agent.create_session()` одржава историју разговора кроз више рунда |

Ове градивне блокове састављају агенте који могу водити природне разговоре, позивати спољне функције и одржавати контекст — основе за напредније агенцијске обрасце у каснијим лекцијама.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Изјава о одрицању одговорности**:  
Овај документ је преведен коришћењем AI услуге за превођење [Co-op Translator](https://github.com/Azure/co-op-translator). Иако настојимо да превод буде тачан, молимо вас да имате у виду да аутоматизовани преводи могу садржати грешке или нетачности. Изворни документ на оригиналном језику треба сматрати ауторитетним извором. За критичне информације препоручује се професионални превод од стране људи. Нисмо одговорни за било каква неспоразума или погрешна тумачења која произилазе из коришћења овог превода.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
